# Day 65 - embeddings and vector search

An embedding turns a document into a direction in space, and retrieval turns a question
into "whose direction is closest". This notebook builds a corpus whose true meaning is
known, trains an embedding on it with a term-count matrix and an SVD, and then measures
what every practical decision costs: normalising or not, how many dimensions, how few
bits, one vector per document or several, and when a second score is worth paying for.

## 1. A corpus with an answer key

Every query gets 10 planted answers, so recall@10 is a fact rather than a taste test.

In [ ]:
import numpy as np
from embeddings import (make_world, train_embedding, truth, unit, topk, tfidf,
                        search, recall_at_k, length_bias, dimension_curve,
                        quantization_table, fusion_table, asymmetry_table,
                        chunking_table, make_long_world, scan_cost, duplicates, check)

world = make_world()
emb = train_embedding(world)
gold = truth(world)
cfg = world['cfg']

base = recall_at_k(search(emb['queries'], emb['docs'], 10), gold)
oracle = recall_at_k(topk(unit(world['qmix']) @ unit(world['mix']).T, 10), gold)
tf, idf = tfidf(world['counts'])
qtf, _ = tfidf(world['qcounts'], idf)
lexical = recall_at_k(topk(unit(qtf) @ unit(tf).T, 10), gold)
print(f"{cfg['docs']:,} documents, {cfg['queries']} queries, {cfg['targets']} planted answers each\n")
print(f"term matching (tf-idf cosine) : recall@10 {lexical:.3f}")
print(f"our embedding ({cfg['dims']} dims)      : recall@10 {base:.3f}")
print(f"the answer key                : recall@10 {oracle:.3f}  (the ceiling)")

20,000 documents, 200 queries, 10 planted answers each

term matching (tf-idf cosine) : recall@10 0.419
our embedding (256 dims)      : recall@10 0.617
the answer key                : recall@10 0.825  (the ceiling)


## 2. A vector is a direction, so the length has to go

In [ ]:
lb = length_bias(world, emb)
print(f"vector length vs document length: r = {lb['norm_vs_length_corr']:.2f}, "
      f"median document {lb['median_length']:.0f} words\n")
for name, r in lb['rows'].items():
    print(f"{name:22} recall@10 {r['recall']:.3f}   "
          f"hits average {r['mean_length_of_hits']:.0f} words")

vector length vs document length: r = 0.92, median document 199 words

cosine (normalised)    recall@10 0.617   hits average 250 words
raw dot product        recall@10 0.092   hits average 1494 words


## 3. The two sides are not symmetric

Normalising the *query* cannot change the ranking; everything else here changes the
answer with no error anywhere.

In [ ]:
for r in asymmetry_table(world, emb):
    print(f"{r['name']:42} recall@10 {r['recall']:.3f}")

both sides normalised (cosine)             recall@10 0.617
query left unnormalised                    recall@10 0.617
documents left unnormalised                recall@10 0.092
query encoded without the idf weights      recall@10 0.629
query encoded with an idf of its own       recall@10 0.609
query with the idf applied twice           recall@10 0.588


## 4. How many dimensions, and how few bits

In [ ]:
curve = dimension_curve(world, emb)
for r in curve:
    print(f"{r['dims']:4} dims  {r['bytes_per_doc']:6,} B/doc  "
          f"{r['gb_per_million']:5.2f} GB per million  recall@10 {r['recall']:.3f}")
best = max(curve, key=lambda r: r['recall'])
print(f"\npeak at {best['dims']} dimensions")

 256 dims   1,024 B/doc   0.95 GB per million  recall@10 0.617
 128 dims     512 B/doc   0.48 GB per million  recall@10 0.678
  64 dims     256 B/doc   0.24 GB per million  recall@10 0.710
  32 dims     128 B/doc   0.12 GB per million  recall@10 0.730
  16 dims      64 B/doc   0.06 GB per million  recall@10 0.732
   8 dims      32 B/doc   0.03 GB per million  recall@10 0.672
   4 dims      16 B/doc   0.01 GB per million  recall@10 0.420

peak at 16 dimensions


In [ ]:
trunc = {'queries': emb['queries'][:, :best['dims']],
         'docs': emb['docs'][:, :best['dims']]}
for label, e in (('full width', emb), (f"truncated to {best['dims']}", trunc)):
    print(label)
    for r in quantization_table(world, e):
        print(f"  {r['name']:28} {r['bytes_per_doc']:8,.1f} B/doc   recall@10 {r['recall']:.3f}")

full width
  float32                       1,024.0 B/doc   recall@10 0.617
  int8                            256.0 B/doc   recall@10 0.618
  binary (1 bit)                   32.0 B/doc   recall@10 0.032
  binary + rescore top 40          34.0 B/doc   recall@10 0.086
truncated to 16
  float32                          64.0 B/doc   recall@10 0.732
  int8                             16.0 B/doc   recall@10 0.732
  binary (1 bit)                    2.0 B/doc   recall@10 0.429
  binary + rescore top 40           2.1 B/doc   recall@10 0.522


## 5. A second score is not automatically better

In [ ]:
for r in fusion_table(world, emb):
    print(f"{r['name']:44} recall@10 {r['recall']:.3f}")

vector score only                            recall@10 0.617
lexical score only (on the same shortlist)   recall@10 0.250
reciprocal rank fusion (k=60)                recall@10 0.457
weighted 80% / 20%                           recall@10 0.629


## 6. One vector per document is a bet about the document

Documents made of five sections about different things; the answer sits in one of them.

In [ ]:
rows = chunking_table(make_long_world())
for r in rows:
    print(f"{r['name']:52} {r['vectors_per_doc']} vec/doc  "
          f"{r['bytes_per_doc']:6,} B/doc  recall@10 {r['recall']:.3f}")

one vector for the whole document                    1 vec/doc     512 B/doc  recall@10 0.100
one vector per section (5), score = best section     5 vec/doc   2,560 B/doc  recall@10 0.584


## 7. What brute force costs

In [ ]:
for r in scan_cost():
    print(f"{r['docs']:9,} docs  {r['gb']:5.2f} GB  {r['ms_per_query']:7.2f} ms/query  "
          f"{r['queries_per_second']:8.0f} queries/s")
print()
print(duplicates(world, emb))

   10,000 docs   0.01 GB     0.84 ms/query      1194 queries/s
  100,000 docs   0.10 GB     9.41 ms/query       106 queries/s
  400,000 docs   0.38 GB    29.24 ms/query        34 queries/s

{'copies': 3, 'slots_taken_for_that_query': 3.0, 'recall_overall': 0.6155}


## 8. Assertions

In [ ]:
check()
print('all assertions passed')

all assertions passed
